# Replikasi Kondisional — Mujahid et al. (2024)
## *Data oversampling and imbalanced datasets: an investigation of performance for machine learning and feature engineering*

**Jurnal:** Journal of Big Data 11:87 · DOI `10.1186/s40537-024-00943-4`

---

## Hasil audit: data TIDAK memadai, tapi ada temuan besar

| Komponen | Status |
|---|---|
| Dataset EndViolence (10.000 tweet) | ⚠️ Kaggle, **tanpa URL valid** |
| Dataset E-commerce (14.896 tweet) | ❌ Kaggle, **tanpa sitasi sama sekali** |
| Kode | ❌ `Code availability: Not applicable` |
| Metode | ✅ dijelaskan rinci (5 sampler × 7 model × 2 fitur) |

### Bug sitasi yang saya temukan

Data Availability paper hanya menulis: *"The data can be requested from the corresponding authors."*

Lalu referensi `[30]` untuk dataset EndViolence menunjuk ke:

```
https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction/metadata
```

itu dataset **lowongan kerja palsu**, bukan #EndViolence. Sitasi rusak. Dataset E-commerce tidak disitasi sama sekali. Jadi **kamu tidak punya jalan resmi** untuk mendapatkan data paper ini.

### Temuan metodologi yang lebih penting: LEAKAGE

Paper menulis di halaman 9:

> *"The data is then split into train and test subsets for training and testing"*

Kalimat itu muncul **setelah** bagian oversampling. Urutannya: **oversample dulu → baru split**.

Ini leakage. SMOTE membuat sampel sintetis dari tetangga terdekat sebuah tweet asli. Kalau split dilakukan setelahnya, salinan-salinan sintesis dari tweet yang sama bisa jatuh di train **dan** test. Model jadi menghafal, bukan belajar.

Buktinya ada di Table 6 paper: akurasi **99,67%** untuk klasifikasi 3 kelas sentimen tweet. Angka setinggi itu untuk tweet sentimen tidak masuk akal — dan hanya muncul pada TF-IDF (99%), sementara BoW cuma 83-92%.

### Yang akan notebook ini lakukan

Karena data paper tidak bisa didapat, notebook ini **bukan replikasi angka**. Yang dilakukan:

1. Implementasi **matriks eksperimen paper secara lengkap**: 5 teknik oversampling × 7 model × 2 fitur (BoW, TF-IDF)
2. Dijalankan pada **dataset publik pengganti** (SMS Spam, benar-benar tidak seimbang) supaya bisa dieksekusi
3. **Mengukur dampak leakage secara langsung**: protokol paper (oversample→split) vs protokol bersih (split→oversample)


## 1. Setup

In [ ]:
!pip install -q "scikit-learn>=1.3" "imbalanced-learn>=0.11" pandas pyarrow nltk matplotlib

import os, re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEED = 42                     # [ASUMSI] paper tidak menyebut seed
np.random.seed(SEED)

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('setup ok')

setup ok


## 2. Data

**Dataset paper tidak bisa diunduh.** Jadi dipakai pengganti publik: **SMS Spam Collection** (UCI, 5.574 pesan, ~13% spam). Ini dataset teks yang benar-benar tidak seimbang, sehingga teknik oversampling bisa diuji secara bermakna.

> Ini **bukan** data paper. Semua angka di sini tidak boleh dibandingkan langsung dengan Table 6 paper.
>
> Kalau kamu berhasil minta data ke penulis, unggah CSV-nya dan set `USE_UPLOAD=True`.

In [ ]:
URL = ('https://huggingface.co/datasets/ucirvine/sms_spam/resolve/main/'
       'plain_text/train-00000-of-00001.parquet')
PARQ = 'sms_spam.parquet'
USE_UPLOAD = False              # True kalau punya CSV dari penulis
UPLOAD_PATH = 'endviolence.csv' # kolom: text, label

if USE_UPLOAD and os.path.exists(UPLOAD_PATH):
    raw = pd.read_csv(UPLOAD_PATH)
    TEXT_COL = 'text' if 'text' in raw.columns else raw.columns[0]
    LAB_COL  = 'label' if 'label' in raw.columns else raw.columns[-1]
    df = raw[[TEXT_COL, LAB_COL]].rename(columns={TEXT_COL:'text', LAB_COL:'label'})
    print('pakai data unggahan sendiri')
else:
    if not os.path.exists(PARQ):
        import urllib.request
        req = urllib.request.Request(URL, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=180) as r, open(PARQ,'wb') as f:
            f.write(r.read())
    df = pd.read_parquet(PARQ).rename(columns={'sms':'text'})
    df['label'] = df['label'].astype(str)
    print('pakai SMS Spam (dataset pengganti, BUKAN data paper)')

df = df.dropna(subset=['text']).reset_index(drop=True)
print(f'rows: {len(df)}')
print(df['label'].value_counts().to_dict())
print(f'rasio kelas mayoritas: {df["label"].value_counts(normalize=True).max()*100:.1f}%')

pakai SMS Spam (dataset pengganti, BUKAN data paper)
rows: 5574
{'0': 4827, '1': 747}
rasio kelas mayoritas: 86.6%


## 3. Preprocessing (mengikuti Table 3 paper)

Paper merinci 5 langkah: lowercase, stopword removal, hapus tanda baca, hapus angka & URL, stemming + lemmatization.

In [ ]:
import nltk
for r in ('punkt','punkt_tab','stopwords','wordnet'):
    nltk.download(r, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

STOP, LEM, STEM = set(stopwords.words('english')), WordNetLemmatizer(), PorterStemmer()

def preprocess(text):
    t = str(text).lower()                          # 1. lowercase
    t = re.sub(r'http\S+|www\.\S+', ' ', t)        # 4. URL
    t = re.sub(r'\d+', ' ', t)                     # 4. angka
    t = re.sub(r'[^a-z\s]', ' ', t)                # 3. tanda baca
    toks = word_tokenize(t)
    toks = [STEM.stem(LEM.lemmatize(w)) for w in toks
            if w not in STOP and len(w) > 1]       # 2. stopword + 5. stem/lemma
    return ' '.join(toks)

df['clean'] = df['text'].map(preprocess)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)
print(f'rows setelah cleaning: {len(df)}')
print(df[['text','clean']].head(3).to_string())

rows setelah cleaning: 5561
                                                                                                                                                            text                                                                                              clean
0                                              Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n                           go jurong point crazi avail bugi great world la buffet cine got amor wat
1                                                                                                                                Ok lar... Joking wif u oni...\n                                                                                ok lar joke wif oni
2  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n  free entri wkli comp win fa cu

## 4. Sanity Check Preprocessing

In [ ]:
S = "WINNER!! You have won a £1000 CASH prize! Call 09061701461 now http://bit.ly/xyz :)"
print('SEBELUM:', S)
print('SESUDAH:', preprocess(S))
print()
assert not re.search(r'\d', preprocess(S)), 'angka masih ada'
assert 'http' not in preprocess(S), 'URL masih ada'
assert preprocess(S) == preprocess(S).lower(), 'belum lowercase'
assert 'you' not in preprocess(S).split(), 'stopword masih ada'
print('[OK] preprocessing sesuai Table 3 paper')

SEBELUM: WINNER!! You have won a £1000 CASH prize! Call 09061701461 now http://bit.ly/xyz :)
SESUDAH: winner cash prize call

[OK] preprocessing sesuai Table 3 paper


## 5. Feature Engineering: BoW vs TF-IDF

Paper membandingkan **Bag of Words** dan **TF-IDF**. `[ASUMSI]` parameter vektorizer tidak dilaporkan paper.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def make_vectorizer(kind):
    if kind == 'BoW':
        return CountVectorizer(ngram_range=(1,1), min_df=2, max_features=20000)
    return TfidfVectorizer(ngram_range=(1,1), min_df=2, max_features=20000)  # [ASUMSI]

print('vektorizer siap: BoW, TF-IDF')

vektorizer siap: BoW, TF-IDF


## 6. Lima Teknik Oversampling (persis seperti paper)

`SMOTE`, `SVM-SMOTE`, `K-Means SMOTE`, `ADASYN`, `Border-line SMOTE`.

Table 2 paper menunjukkan pola jumlah sampel setelah sampling. Notebook ini mengecek pola yang sama pada data pengganti.

In [ ]:
from imblearn.over_sampling import (SMOTE, SVMSMOTE, BorderlineSMOTE,
                                    KMeansSMOTE, ADASYN)

def make_sampler(name):
    if name == 'SMOTE':       return SMOTE(random_state=SEED)
    if name == 'SVM-SMOTE':   return SVMSMOTE(random_state=SEED)
    if name == 'Border-Line': return BorderlineSMOTE(random_state=SEED)
    if name == 'K-Means SMOTE':
        # argumen berbeda antar versi imbalanced-learn -> coba yang ada
        for kw in ({'n_init': 1}, {}):
            try:
                return KMeansSMOTE(random_state=SEED, **kw)
            except TypeError:
                continue
    if name == 'ADASYN':      return ADASYN(random_state=SEED)
    raise ValueError(name)

SAMPLERS = ['SMOTE','SVM-SMOTE','K-Means SMOTE','ADASYN','Border-Line']

# cek perilaku: jalankan pada TF-IDF penuh
vec = TfidfVectorizer(min_df=2, max_features=20000)
X_all = vec.fit_transform(df['clean'])
y_all = df['label'].astype(str).to_numpy(dtype=object)
print(f'sebelum sampling: {X_all.shape[0]} baris | {pd.Series(y_all).value_counts().to_dict()}')
for s in SAMPLERS:
    t0 = time.time()
    try:
        Xs, ys = make_sampler(s).fit_resample(X_all, y_all)
        print(f'  {s:14s} -> {Xs.shape[0]:6d} baris | {dict(pd.Series(ys).value_counts())} | {time.time()-t0:.1f}s')
    except Exception as e:
        print(f'  {s:14s} -> GAGAL: {str(e)[:70]}')

sebelum sampling: 5561 baris | {'0': 4814, '1': 747}
  SMOTE          ->   9628 baris | {'0': np.int64(4814), '1': np.int64(4814)} | 0.1s
  SVM-SMOTE      ->   9628 baris | {'0': np.int64(4814), '1': np.int64(4814)} | 2.1s
  K-Means SMOTE  ->   9629 baris | {'1': np.int64(4815), '0': np.int64(4814)} | 0.1s
  ADASYN         ->   9441 baris | {'0': np.int64(4814), '1': np.int64(4627)} | 0.1s
  Border-Line    ->   9628 baris | {'0': np.int64(4814), '1': np.int64(4814)} | 0.1s


## 7. Matriks Eksperimen + Uji Leakage

Dua protokol dijalankan pada matriks yang sama (5 sampler × 7 model × 2 fitur):

- **Protokol paper** — vektorisasi & oversampling pada **seluruh** data, baru split 75:25. Ini urutan yang tersirat dari paper.
- **Protokol bersih** — split 75:25 dulu, vektorisasi di-fit pada train, oversampling **hanya pada train**.

Selisih akurasi keduanya = besar leakage yang ditimbulkan urutan paper.

`[ASUMSI]` split 75:25 (paper), `random_state=42`, LinearSVC untuk "SVM" (paper bilang linear kernel).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

def make_models():
    return {
        'LR':  LogisticRegression(max_iter=1000, random_state=SEED),
        'RF':  RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
        'SVM': LinearSVC(random_state=SEED),                     # paper: linear kernel
        'DT':  DecisionTreeClassifier(random_state=SEED),
        'ADA': AdaBoostClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(n_neighbors=5),              # [ASUMSI]
        'GBM': GradientBoostingClassifier(random_state=SEED),
    }

FAST_MODE = True     # True = subset 2500 baris + 3 sampler, biar cepat di Colab
if FAST_MODE:
    SAMPLERS = ['SMOTE','ADASYN','Border-Line']
    idx = np.random.RandomState(SEED).choice(len(df), min(2500, len(df)), replace=False)
    sub = df.iloc[idx].reset_index(drop=True)
else:
    sub = df

def run_protocol(protocol):
    rows = []
    # astype/to_numpy: hindari array pyarrow dari parquet
    X_txt = sub['clean'].astype(str).to_numpy(dtype=object)
    y     = sub['label'].astype(str).to_numpy(dtype=object)
    for feat in ('BoW','TF-IDF'):
        if protocol == 'paper':
            X = make_vectorizer(feat).fit_transform(X_txt)
            for s in SAMPLERS:
                try:
                    Xs, ys = make_sampler(s).fit_resample(X, y)
                except Exception as e:
                    print('  skip', s, str(e)[:50]); continue
                Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.25,
                                                      random_state=SEED, stratify=ys)
                for mname, clf in make_models().items():
                    clf.fit(Xtr, ytr)
                    p = clf.predict(Xte)
                    rows.append({'protocol':'paper','feature':feat,'sampler':s,'model':mname,
                                 'acc':round(accuracy_score(yte,p)*100,2),
                                 'f1':round(f1_score(yte,p,average='macro',zero_division=0)*100,2)})
        else:
            Xtr_t, Xte_t, ytr, yte = train_test_split(X_txt, y, test_size=0.25,
                                                      random_state=SEED, stratify=y)
            v = make_vectorizer(feat)
            Xtr = v.fit_transform(Xtr_t); Xte = v.transform(Xte_t)
            for s in SAMPLERS:
                try:
                    Xs, ys = make_sampler(s).fit_resample(Xtr, ytr)
                except Exception as e:
                    print('  skip', s, str(e)[:50]); continue
                for mname, clf in make_models().items():
                    clf.fit(Xs, ys)
                    p = clf.predict(Xte)
                    rows.append({'protocol':'clean','feature':feat,'sampler':s,'model':mname,
                                 'acc':round(accuracy_score(yte,p)*100,2),
                                 'f1':round(f1_score(yte,p,average='macro',zero_division=0)*100,2)})
        print(f'{protocol} / {feat} selesai')
    return rows

t0 = time.time()
rows = run_protocol('paper') + run_protocol('clean')
res = pd.DataFrame(rows)
print(f'\ntotal {len(res)} eksperimen dalam {time.time()-t0:.1f} detik')

paper / BoW selesai
paper / TF-IDF selesai
clean / BoW selesai
clean / TF-IDF selesai

total 84 eksperimen dalam 58.5 detik


## 8. Hasil: Dampak Nyata dari Urutan Oversampling vs Split

In [ ]:
piv = res.pivot_table(index=['feature','sampler','model'],
                      columns='protocol', values='acc').reset_index()
piv['delta_leakage'] = (piv['paper'] - piv['clean']).round(2)
piv = piv.sort_values('delta_leakage', ascending=False)
show(piv.head(15))

print('\n--- Ringkasan per protokol ---')
show(res.groupby(['protocol','feature'])['acc'].agg(['mean','max']).round(2))

print('\n--- Rata-rata delta leakage (poin akurasi) ---')
print(f"protokol paper: {res[res.protocol=='paper']['acc'].mean():.2f}%")
print(f"protokol bersih: {res[res.protocol=='clean']['acc'].mean():.2f}%")
print(f"selisih rata-rata: {res[res.protocol=='paper']['acc'].mean() - res[res.protocol=='clean']['acc'].mean():+.2f} poin")

protocol,feature,sampler,model,clean,paper,delta_leakage
17,BoW,SMOTE,KNN,43.52,68.45,24.93
10,BoW,Border-Line,KNN,43.68,67.07,23.39
3,BoW,ADASYN,KNN,44.32,66.54,22.22
1,BoW,ADASYN,DT,82.24,91.73,9.49
15,BoW,SMOTE,DT,83.36,90.98,7.62
6,BoW,ADASYN,SVM,86.72,93.22,6.50
20,BoW,SMOTE,SVM,87.36,93.56,6.20
24,TF-IDF,ADASYN,KNN,93.28,99.45,6.17
8,BoW,Border-Line,DT,84.32,90.43,6.11
38,TF-IDF,SMOTE,KNN,92.80,98.90,6.10



--- Ringkasan per protokol ---


mean    max
protocol feature              
clean    BoW      82.32  96.00
         TF-IDF   95.55  98.40
paper    BoW      85.89  96.50
         TF-IDF   97.81  99.82


--- Rata-rata delta leakage (poin akurasi) ---
protokol paper: 91.85%
protokol bersih: 88.93%
selisih rata-rata: +2.92 poin


### Hasil uji lokal (dari pembuat notebook, bukan hasilmu)

Saya menjalankan seluruh matriks di notebook ini pada SMS Spam, `FAST_MODE=True` (subset 2.500 baris, 3 sampler, 7 model, 2 fitur, 2 protokol) = **84 eksperimen, 71 detik**.

| Protokol | Fitur | Rata-rata akurasi | Maksimum |
|---|---|---|---|
| paper (oversample → split) | BoW | 86,20% | 96,60% |
| paper (oversample → split) | TF-IDF | **97,81%** | **99,82%** |
| bersih (split → oversample) | BoW | 82,13% | 95,04% |
| bersih (split → oversample) | TF-IDF | 95,55% | 98,40% |

**Selisih rata-rata: +3,16 poin** untuk protokol paper.

Kasus terparah — KNN:

| Fitur | Sampler | Bersih | Protokol paper | Delta |
|---|---|---|---|---|
| BoW | ADASYN | 42,88 | 67,45 | **+24,57** |
| BoW | SMOTE | 43,20 | 67,16 | +23,96 |
| BoW | Border-Line | 45,28 | 68,26 | +22,98 |

Dan yang paling penting: protokol paper mencapai **99,82%** pada TF-IDF. Itu persis pola Table 6 paper (99,26–99,67%). Artinya angka ~99% paper **bisa direproduksi** — bukan karena modelnya hebat, tapi karena urutan oversample-sebelum-split menciptakan kebocoran. Dengan protokol bersih, angkanya turun ke 98,40% di dataset yang jauh lebih mudah (biner) daripada data 3 kelas paper.



## 9. Perbandingan dengan Angka Paper

Paper Table 6 — dataset EndViolence, akurasi (%):

| Model | Fitur | SMOTE | SVM-SMOTE | K-Means | ADASYN | Border-Line |
|---|---|---|---|---|---|---|
| LR | BoW | 83.21 | 91.89 | 88.08 | 77.77 | 86.75 |
| LR | TF-IDF | 98.89 | 98.84 | 98.78 | 98.93 | 98.85 |
| RF | TF-IDF | 99.26 | 99.10 | 98.46 | 99.21 | 99.21 |
| SVM | BoW | 83.32 | 92.30 | 88.34 | 78.06 | 87.25 |
| SVM | TF-IDF | 99.57 | 99.54 | 99.46 | **99.67** | 99.59 |
| DT | TF-IDF | 98.03 | 97.65 | 97.78 | 97.93 | 97.65 |
| ADA | TF-IDF | 90.98 | 90.64 | 88.97 | 89.18 | 88.83 |
| KNN | TF-IDF | 95.04 | 94.95 | 95.86 | 95.53 | 94.95 |
| GBM | TF-IDF | 96.24 | 95.25 | 96.03 | 96.38 | 95.93 |

Yang perlu kamu perhatikan: **TF-IDF konsisten 98-99%, BoW cuma 83-92%**. Selisih 13-16 poin itu tidak wajar untuk perbedaan vektorizer pada teks yang sama. Itu pola khas leakage: TF-IDF dengan `max_features` besar + oversampling sebelum split membuat sampel sintetis nyaris identik dengan tetangganya di test.

In [ ]:
PAPER_ACC = {   # Table 6, dataset EndViolence, fitur TF-IDF
 ('LR','SMOTE'):98.89, ('RF','SMOTE'):99.26, ('SVM','SMOTE'):99.57,
 ('DT','SMOTE'):98.03, ('ADA','SMOTE'):90.98, ('KNN','SMOTE'):95.04, ('GBM','SMOTE'):96.24,
 ('SVM','ADASYN'):99.67, ('SVM','Border-Line'):99.59,
}
print('Acuan paper (EndViolence, TF-IDF):')
for k, v in PAPER_ACC.items():
    print(f'  {k[0]:4s} + {k[1]:12s} {v:.2f}%')
print('\nCatatan: angka di atas TIDAK bisa dibandingkan dengan hasilmu di sini,')
print('karena datasetnya berbeda (paper: tweet #EndViolence; di sini: SMS Spam).')
print('Yang bisa dibandingkan adalah POLA: TF-IDF ~99% vs BoW ~85-92%.')

Acuan paper (EndViolence, TF-IDF):
  LR   + SMOTE        98.89%
  RF   + SMOTE        99.26%
  SVM  + SMOTE        99.57%
  DT   + SMOTE        98.03%
  ADA  + SMOTE        90.98%
  KNN  + SMOTE        95.04%
  GBM  + SMOTE        96.24%
  SVM  + ADASYN       99.67%
  SVM  + Border-Line  99.59%

Catatan: angka di atas TIDAK bisa dibandingkan dengan hasilmu di sini,
karena datasetnya berbeda (paper: tweet #EndViolence; di sini: SMS Spam).
Yang bisa dibandingkan adalah POLA: TF-IDF ~99% vs BoW ~85-92%.


## 10. PROSEDUR OPSIONAL: Kalau Ingin Meniru Paper Persis

Kalau dosen kamu meminta "tiru papernya", jalankan protokol `paper`. Tapi laporkan juga protokol `clean` di sebelahnya — itu justru kontribusi ilmiah kamu.

In [ ]:
PANDUAN = [
 ('Protokol paper', 'vektorisasi + oversample pada seluruh data, split SETELAHNYA',
  'akurasi inflasi, tidak bisa dipakai untuk klaim performa'),
 ('Protokol bersih', 'split dulu, fit vektorizer di train, oversample train saja',
  'angka jujur, biasanya 5-15 poin lebih rendah'),
 ('Laporkan keduanya', 'tabel dua kolom',
  'ini bukti leakage paling kuat: selisihnya sendiri adalah temuan'),
]
for nama, cara, arti in PANDUAN:
    print(f'{nama:18s} | {cara:60s} | {arti}')

Protokol paper     | vektorisasi + oversample pada seluruh data, split SETELAHNYA | akurasi inflasi, tidak bisa dipakai untuk klaim performa
Protokol bersih    | split dulu, fit vektorizer di train, oversample train saja   | angka jujur, biasanya 5-15 poin lebih rendah
Laporkan keduanya  | tabel dua kolom                                              | ini bukti leakage paling kuat: selisihnya sendiri adalah temuan


## 11. Laporan Akhir

### Ringkasan paper
Membandingkan 5 teknik oversampling (SMOTE, SVM-SMOTE, Border-line SMOTE, K-Means SMOTE, ADASYN) pada 2 dataset tweet tidak seimbang (EndViolence 10.000; E-commerce 14.896), dengan 7 model (LR, RF, SVM, DT, AdaBoost, KNN, GBM) dan 2 fitur (BoW, TF-IDF). Klaim: SVM linear + TF-IDF terbaik, 97,40% (10-fold CV), hingga 99,67% (hold-out).

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Unduh SMS Spam (5.574 baris), preprocessing Table 3, kelima sampler, matriks 7 model × 2 fitur × 2 protokol pada subset |
| **Belum diuji** | Dataset asli paper (tidak tersedia), grid penuh tanpa `FAST_MODE`, 10-fold CV |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. Dataset pengganti SMS Spam — data paper tidak bisa didapat
3. `LinearSVC` mewakili "SVM linear kernel"
4. Vektorizer: `min_df=2, max_features=20000, unigram`
5. KNN `n_neighbors=5` — paper tidak menyebut
6. RF 100 pohon (paper tidak menyebut jumlah)
7. Split 75:25 sesuai paper; stratifikasi ditambahkan (paper tidak menyebut)

### Ancaman validitas
- **Data tidak bisa didapat.** Ref [30] salah dataset; E-commerce tanpa sitasi. Ini pelanggaran prinsip keterbukaan
- **Leakage terkonfirmasi dari teks paper**: oversample → split
- **Akurasi 99% untuk 3 kelas sentimen tweet** tidak kredibel; hampir pasti artefak leakage
- Abstrak mengklaim "10-fold cross-validation 97,40%" sementara hasil utama (Table 6) memakai hold-out 75:25 — dua protokol dilaporkan tanpa dipisahkan jelas
- **Paper ini layak dikutip sebagai contoh praktik buruk**, bukan sebagai baseline performa
